## Data Preparation

In [1]:
import keras
import keras.backend as K

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



2023-11-09 11:07:54.814728: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
from sklearn import preprocessing

train_df = pd.read_csv('./Dataset/df_train.txt',sep=" ", header=None)
train_df.drop(train_df.columns[[26, 27]], axis=1, inplace=True)
train_df.columns = ['id', 'cycle', 'setting1', 'setting2', 'setting3', 's1', 's2', 's3',
                     's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14',
                     's15', 's16', 's17', 's18', 's19', 's20', 's21']
train_df.head(5)

,id,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


### Turbofan Train Set

In [ ]:
# Generate column RUL 
rul = pd.DataFrame(train_df.groupby('id')['cycle'].max()).reset_index()
rul.columns = ['id', 'max']
train_df = train_df.merge(rul, on=['id'], how='left')
train_df['RUL'] = train_df['max'] - train_df['cycle']

train_df['RUL'] = np.where((train_df.RUL > 130),130.0,train_df.RUL)

train_df.drop('max', axis=1, inplace=True)

train_df['cycle_norm'] = train_df['cycle']
cols_normalize = train_df.columns.difference(['id','cycle','RUL','label1','label2'])
min_max_scaler = preprocessing.MinMaxScaler()
norm_train_df = pd.DataFrame(min_max_scaler.fit_transform(train_df[cols_normalize]), 
                             columns=cols_normalize, 
                             index=train_df.index)
join_df = train_df[train_df.columns.difference(cols_normalize)].join(norm_train_df)
train_df = join_df.reindex(columns = train_df.columns)

train_df[train_df["id"] == 2].tail()

,id,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,...,s14,s15,s16,s17,s18,s19,s20,s21,RUL,cycle_norm
474,2,283,0.764368,0.666667,0.0,0.0,0.774096,0.675605,0.800473,0.0,...,0.386572,0.750289,0.0,0.833333,0.0,0.0,0.217054,0.195526,4.0,0.781163
475,2,284,0.465517,0.583333,0.0,0.0,0.813253,0.660780,0.806718,0.0,...,0.345185,0.785687,0.0,0.583333,0.0,0.0,0.069767,0.311240,3.0,0.783934
476,2,285,0.459770,0.833333,0.0,0.0,0.740964,0.562459,0.825793,0.0,...,0.334864,0.766833,0.0,0.666667,0.0,0.0,0.193798,0.305579,2.0,0.786704
477,2,286,0.442529,0.250000,0.0,0.0,0.671687,0.710486,0.798785,0.0,...,0.361389,0.647557,0.0,0.583333,0.0,0.0,0.147287,0.169428,1.0,0.789474
478,2,287,0.471264,1.000000,0.0,0.0,0.795181,0.816656,0.820223,0.0,...,0.390288,0.814159,0.0,0.833333,0.0,0.0,0.224806,0.263187,0.0,0.792244


### Turbofan Test Set

In [4]:


# read test data 
test_df = pd.read_csv('Dataset/df_test.txt', sep=" ", header=None)
test_df.drop(test_df.columns[[26, 27]], axis=1, inplace=True)
test_df.columns = ['id', 'cycle', 'setting1', 'setting2', 'setting3', 's1', 's2', 's3',
                     's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14',
                     's15', 's16', 's17', 's18', 's19', 's20', 's21']

test_df['cycle_norm'] = test_df['cycle']
norm_test_df = pd.DataFrame(min_max_scaler.transform(test_df[cols_normalize]), 
                            columns=cols_normalize, 
                            index=test_df.index)
test_join_df = test_df[test_df.columns.difference(cols_normalize)].join(norm_test_df)
test_df = test_join_df.reindex(columns = test_df.columns)
test_df = test_df.reset_index(drop=True)

# read truth data 
tru_df = pd.read_csv('Dataset/df_true.txt', sep=" ", header=None)
tru_df.drop(tru_df.columns[[1]], axis=1, inplace=True)
tru = tru_df.iloc[:,0].values
rul = pd.DataFrame(test_df.groupby('id')['cycle'].max()).reset_index()
rul.columns = ['id', 'max']
tru_df.columns = ['tam']
tru_df['id'] = tru_df.index + 1
tru_df['max'] = rul['max'] + tru_df['tam']
tru_df.drop('tam', axis=1, inplace=True)

# generate RUL for test data
test_df = test_df.merge(tru_df, on=['id'], how='left')
test_df['RUL'] = test_df['max'] - test_df['cycle']
test_df0 = test_df.copy()

#RUL piece-wise 130
test_df['RUL'] = np.where((test_df0.RUL > 130),130.0,test_df0.RUL)

test_df.drop('max', axis=1, inplace=True)

test_df[test_df["id"] == 1].head()

,id,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,...,s14,s15,s16,s17,s18,s19,s20,s21,cycle_norm,RUL
0,1,1,0.632184,0.750000,0.0,0.0,0.545181,0.310661,0.269413,0.0,...,0.132160,0.308965,0.0,0.333333,0.0,0.0,0.558140,0.661834,0.00000,130.0
1,1,2,0.344828,0.250000,0.0,0.0,0.150602,0.379551,0.222316,0.0,...,0.204768,0.213159,0.0,0.416667,0.0,0.0,0.682171,0.686827,0.00277,130.0
2,1,3,0.517241,0.583333,0.0,0.0,0.376506,0.346632,0.322248,0.0,...,0.155640,0.458638,0.0,0.416667,0.0,0.0,0.728682,0.721348,0.00554,130.0
3,1,4,0.741379,0.500000,0.0,0.0,0.370482,0.285154,0.408001,0.0,...,0.170090,0.257022,0.0,0.250000,0.0,0.0,0.666667,0.662110,0.00831,130.0
4,1,5,0.580460,0.500000,0.0,0.0,0.391566,0.352082,0.332039,0.0,...,0.152751,0.300885,0.0,0.166667,0.0,0.0,0.658915,0.716377,0.01108,130.0


In [5]:
print(max(test_df0.RUL.values))
print(min(test_df0.RUL.values))

340
7


### Making all the sequences having the same lenght (add zero to the shorter ones)

In [6]:
def sequence(df, max_seq, mask=0):
    return np.pad(df.values, ((0, max_seq- df.values.shape[0]), (0,0)), 
                  "constant", constant_values=mask)
  
def engine(df, cols, max_batch, mask=0):
    return np.array([sequence(df[df['id'] == batch_id][cols], max_batch, mask=mask) 
                     for batch_id in df['id'].unique()])
  
max_batch = train_df['id'].value_counts().max()
train_cols = ['s' + str(i) for i in [2, 3, 4, 7, 8, 9, 11, 12, 13, 14, 15, 17, 20, 21]] + ['cycle_norm']

test_cols = ["RUL"]

X = engine(train_df, train_cols, max_batch)
Y = engine(train_df, test_cols, max_batch)

### Split into train, validation and test sets

In [ ]:
from sklearn.model_selection import train_test_split

# Split into train and validation
train_X = X[:70,:,:]
train_Y = Y[:70,:,:]
val_X   = X[70:,:,:]
val_Y   = Y[70:,:,:]

test_X = engine(test_df, train_cols, max_batch)
test_Y = engine(test_df, test_cols, max_batch)

train_Y_ln = np.concatenate((train_Y, np.ones(train_Y.shape)), axis=2)
val_Y_ln = np.concatenate((val_Y, np.ones(val_Y.shape)), axis=2)
test_Y_ln = np.concatenate((test_Y, np.ones(test_Y.shape)), axis=2)

print("Train data set:\n", "  X:", train_X.shape,  "\n  Y_ln:", train_Y_ln.shape)
print("\n Validation data set:\n", "  X:", val_X.shape, "\n  Y_ln:", val_Y_ln.shape)
print("\n Test data set:\n", "  X:", test_X.shape, "\n  Y_ln:", test_Y_ln.shape)

Train data set:
   X: (70, 362, 15) 
  Y_ln: (70, 362, 2)

 Validation data set:
   X: (30, 362, 15) 
  Y_ln: (30, 362, 2)

 Test data set:
   X: (100, 362, 15) 
  Y_ln: (100, 362, 2)


## Log - normal distribution ##

In [8]:

from scipy.stats import lognorm
from keras.layers import Dense, Activation, Layer, Input
from keras.models import Sequential, Model
from keras.layers import Masking
from keras.layers import Dense, LSTM, TimeDistributed, Lambda
from keras.callbacks import EarlyStopping, ModelCheckpoint, TerminateOnNaN
from keras import optimizers
import tensorflow_probability as tfp
import tensorflow as tf
from tensorflow import keras

from scipy import special
from numpy.lib.scimath import logn
from math import e


def log_normal_layer(x):
    num_dims = len(x.get_shape())
    muy, sigma = tf.unstack(x, num=2, axis=-1)
    
    ######
    #adjust sigma
    y=tf.constant(0.5)
    result = tf.where(sigma < y, tf.add(sigma,0), tf.constant(0.5))
    sigma = result
    
    #######
    muy = tf.expand_dims(muy, -1)
    sigma = tf.expand_dims(sigma, -1)
    sigma = tf.nn.elu(sigma) + 1
    out_tensor = tf.concat((muy, sigma), axis=num_dims-1)

    return out_tensor

def mdn_log_cost(y_actual, y_pred):
    n, p = tf.unstack(y_pred, num=2, axis=-1)
    mu = tf.expand_dims(n, -1)
    sigma = tf.expand_dims(p, -1)
    dist = tfp.distributions.LogNormal(loc=mu, scale=sigma)
    return tf.reduce_mean(-dist.log_prob(y_actual+0.001))

def cdf_lognorm(x, mu, sig):
    return (0.5 + 0.5*special.erf((np.log(x) -mu)/(2**0.5*sig) ))

def Metric_eval(function):
    train_set = test_X
    test_set = test_Y

    name_set = "Test"
    pred = model.predict(train_set)
    n, p = tf.unstack(pred, num=2, axis=-1)
    mu = tf.expand_dims(n, -1)
    sigma = tf.expand_dims(p, -1)
    y_pred = function(mu, sigma).flatten()
            
    l = test_set[:,:,0].flatten()
       
    y_true = np.ma.compressed(np.ma.masked_where(l==0, l))
    y_pred = np.ma.compressed(np.ma.masked_where(l==0, y_pred))

    kk = np.diff(test_df0['RUL'].values)
    ii = np.where(kk > 0)[0]
    index = np.append(ii, 13095)
    e_130 = y_true[index] - y_pred[index]
    il0 = np.where(e_130 <= 0)[0]
    s_130 = np.exp(e_130/13) - 1
    s_130[il0] = np.exp(-e_130[il0]/10) - 1
    score_130 = sum(s_130)
    
    a = np.zeros(100)
    ii = np.where((e_130 <13)&(e_130>-10))[0]
    a[ii] = 1
    acc_130 = sum(a)/100

    e = truth - y_pred[index]
    il0 = np.where(e <= 0)[0]
    s = np.exp(e/13) - 1
    s[il0] = np.exp(-e[il0]/10) - 1
    score = sum(s)
    
    a = np.zeros(100)
    ii = np.where((e <13)&(e>-10))[0]
    a[ii] = 1
    acc = sum(a)/100
    mse = mean_squared_error(truth, y_pred[index])
    mse_130 = mean_squared_error(y_true[index], y_pred[index])
    rmse_130 = sqrt(mean_squared_error(y_true[index], y_pred[index]))
    rmse = sqrt(mean_squared_error(truth, y_pred[index]))
    
    ms = np.exp(mu).flatten()
    sigs = np.exp(sigma).flatten()
    ms = np.ma.compressed(np.ma.masked_where(l==0, ms))[index]
    sigs = np.ma.compressed(np.ma.masked_where(l==0, sigs))[index]

    mincon = ms/(sigs**2)
    maxcon = ms*(sigs**2)
    interval =maxcon - mincon
    
    ytrue = y_true[index]

    be = np.zeros(len(mincon))
    be_130 = np.zeros(len(mincon))
    for ih in np.arange(len(mincon)):
        if (truth[ih] <= maxcon[ih]) & (truth[ih] >= mincon[ih]):
            be[ih] = 1
        if (ytrue[ih] <= maxcon[ih]) & (truth[ih] >= mincon[ih]):
            be_130[ih] = 1
    interval = maxcon-mincon
    NMPIW = sum(interval)/(len(mincon)*(max(truth)-min(truth)))
    NMPIW_130 = sum(interval)/(len(mincon)*(max(ytrue)-min(ytrue)))
    
    print('RMSE when truncate RUL 130:', rmse_130)
    print('RMSE when non-truncate RUL 130:',rmse)
    print('MSE when truncate RUL 130:', mse_130)
    print('MSE when non-truncate RUL 130:', mse)
    print('Score when truncate RUL 130:', score_130)
    print('Score when non-truncate RUL 130:', score)
    print('accuracy when truncate RUL 130:', acc_130)
    print('accuracy when non-truncate RUL 130:',acc)
    print('RUL_true (truncated 130) belongs to 95 interval:', sum(be_130)/len(be))
    print('RUL_true (non truncated 130) belongs to 95 interval:', sum(be)/len(be))
    print('confidence interval wide/ rang RUL (truncated 130):', NMPIW_130 )
    print('confidence interval wide/ rang RUL (non truncated 130):', NMPIW )
    return (rmse_130, rmse, mse_130, mse, score_130, score, acc_130, acc, sum(be_130)/len(be), sum(be)/len(be), NMPIW_130, NMPIW)


In [9]:
def Metric_all_eval(function):
    train_set = test_X
    test_set = test_Y

    name_set = "Test"
    pred = model.predict(train_set)
    n, p = tf.unstack(pred, num=2, axis=-1)
    
    # Add one dimension to make the right shape
    mu = tf.expand_dims(n, -1)
    sigma = tf.expand_dims(p, -1)
    y_pred = function(mu, sigma).flatten()
            
    l = test_set[:,:,0].flatten()
        # To validate we remove the right padding
    y_true = np.ma.compressed(np.ma.masked_where(l==0, l))
    y_pred = np.ma.compressed(np.ma.masked_where(l==0, y_pred))

    kk = np.diff(test_df0['RUL'].values)
    ii = np.where(kk > 0)[0]
    index = np.append(ii, 13095)
#print(y_true[index])
#print(index)
#print(y_pred[index])

    e_130 = y_true - y_pred
    il0 = np.where(e_130 <= 0)[0]
    s_130 = np.exp(e_130/13) - 1
    s_130[il0] = np.exp(-e_130[il0]/10) - 1
    score_130 = sum(s_130)
    
    a = np.zeros(len(e_130))
    ii = np.where((e_130 <13)&(e_130>-10))[0]
    a[ii] = 1
    acc_130 = sum(a)/len(e_130)
    
    truth = test_df0['RUL'].values
    e = truth - y_pred
    il0 = np.where(e <= 0)[0]
    s = np.exp(e/13) - 1
    s[il0] = np.exp(-e[il0]/10) - 1
    score = sum(s)
    
    a = np.zeros(len(e_130))
    ii = np.where((e <13)&(e>-10))[0]
    a[ii] = 1
    acc = sum(a)/100
    mse = mean_squared_error(truth, y_pred)
    mse_130 = mean_squared_error(y_true, y_pred)
    rmse_130 = sqrt(mean_squared_error(y_true, y_pred))
    rmse = sqrt(mean_squared_error(truth, y_pred))
    
    ms = np.exp(mu).flatten()
    sigs = np.exp(sigma).flatten()
    ms = np.ma.compressed(np.ma.masked_where(l==0, ms))
    sigs = np.ma.compressed(np.ma.masked_where(l==0, sigs))

    mincon = ms/(sigs**2)
    maxcon = ms*(sigs**2)
    interval =maxcon - mincon
    
    ytrue = y_true

    be = np.zeros(len(mincon))
    be_130 = np.zeros(len(mincon))
    for ih in np.arange(len(mincon)):
        if (truth[ih] <= maxcon[ih]) & (truth[ih] >= mincon[ih]):
            be[ih] = 1
        if (ytrue[ih] <= maxcon[ih]) & (truth[ih] >= mincon[ih]):
            be_130[ih] = 1
    interval = maxcon-mincon
    NMPIW = sum(interval)/(len(mincon)*(max(truth)-min(truth)))
    NMPIW_130 = sum(interval)/(len(mincon)*(max(ytrue)-min(ytrue)))
    
    print('RMSE when truncate RUL 130:', rmse_130)
    print('RMSE when non-truncate RUL 130:',rmse)
    print('MSE when truncate RUL 130:', mse_130)
    print('MSE when non-truncate RUL 130:', mse)
    print('Score when truncate RUL 130:', score_130)
    print('Score when non-truncate RUL 130:', score)
    print('accuracy when truncate RUL 130:', acc_130)
    print('accuracy when non-truncate RUL 130:',acc)
    print('RUL_true (truncated 130) belongs to 95 interval:', sum(be_130)/len(be))
    print('RUL_true (non truncated 130) belongs to 95 interval:', sum(be)/len(be))
    print('confidence interval wide/ rang RUL (truncated 130):', NMPIW_130 )
    print('confidence interval wide/ rang RUL (non truncated 130):', NMPIW )
    return (rmse_130, rmse, mse_130, mse, score_130, score, acc_130, acc, sum(be_130)/len(be), sum(be)/len(be), NMPIW_130, NMPIW)


In [10]:

LogNormal_LSTM_path = "chinhsig_RUL130_log_normal_LSTM_train70"
# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', 
                               min_delta=0, 
                               patience=50, 
                               verbose=0, 
                               mode='min')
checkpoint = ModelCheckpoint(LogNormal_LSTM_path, 
                             monitor='val_loss', 
                             save_best_only=True, 
                             save_weights_only=True, 
                             mode='min', 
                             verbose=0)
max_batch_len = train_X.shape[1]
nb_features = train_X.shape[2]
nb_out = train_Y.shape[1]

learning_rate = 0.001 #0.01
adamOptimizer = optimizers.Adam(learning_rate=learning_rate)
# Define inputs with predefined shape
model = Sequential()

model.add(Masking(mask_value=0.,
                  input_shape=(max_batch_len, nb_features)))
model.add(LSTM(
         input_shape=(None, nb_features),
         units=100,
         recurrent_dropout=0.2,
         return_sequences=True))
model.add(LSTM(
          units=50,
          recurrent_dropout=0.2,
          return_sequences=True))

model.add(TimeDistributed(Dense(2)))
model.add(Lambda(log_normal_layer))
#y_real = Input(shape=input_shape)
# Construct model
#model = Model(inputs=inputs , outputs=distribution_outputs)
print(model.summary())

model.compile(loss= mdn_log_cost, optimizer=adamOptimizer)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 masking (Masking)           (None, 362, 15)           0         
                                                                 
 lstm (LSTM)                 (None, 362, 100)          46400     
                                                                 
 lstm_1 (LSTM)               (None, 362, 50)           30200     
                                                                 
 time_distributed (TimeDist  (None, 362, 2)            102       
 ributed)                                                        
                                                                 
 lambda (Lambda)             (None, 362, 2)            0         
                                                                 
Total params: 76702 (299.62 KB)
Trainable params: 76702 (299.62 KB)
Non-trainable params: 0 (0.00 Byte)
__________________

In [11]:
history = model.fit(train_X, train_Y, epochs=500, batch_size=1, 
                    validation_data=(val_X, val_Y), shuffle=False, verbose=2,
                    callbacks = [early_stopping, checkpoint, TerminateOnNaN()])

# list all data in history
print(history.history.keys())

Epoch 1/500
70/70 - 35s - loss: 5.1111 - val_loss: 4.0395 - 35s/epoch - 493ms/step
Epoch 2/500
70/70 - 28s - loss: 1.7884 - val_loss: 2.6939 - 28s/epoch - 397ms/step
Epoch 3/500
70/70 - 29s - loss: 1.3993 - val_loss: 1.8349 - 29s/epoch - 409ms/step
Epoch 4/500
70/70 - 28s - loss: 1.1773 - val_loss: 1.6161 - 28s/epoch - 397ms/step
Epoch 5/500
70/70 - 28s - loss: 1.1061 - val_loss: 1.5047 - 28s/epoch - 397ms/step
Epoch 6/500
70/70 - 28s - loss: 1.1228 - val_loss: 2.0125 - 28s/epoch - 395ms/step
Epoch 7/500
70/70 - 45s - loss: 1.1690 - val_loss: 1.6358 - 45s/epoch - 650ms/step
Epoch 8/500



KeyboardInterrupt



Epoch 258/500
70/70 - 27s - loss: -5.0598e-01 - val_loss: -8.6018e-02
Epoch 259/500
70/70 - 27s - loss: -3.4607e-01 - val_loss: -1.0754e-01
Epoch 260/500
70/70 - 26s - loss: -4.1252e-01 - val_loss: 0.3682
Epoch 261/500
70/70 - 26s - loss: -2.8620e-01 - val_loss: -1.6914e-02
Epoch 262/500
70/70 - 25s - loss: -2.2453e-01 - val_loss: 0.3034
Epoch 263/500
70/70 - 25s - loss: -3.1965e-01 - val_loss: -1.0229e-01
dict_keys(['loss', 'val_loss'])


In [ ]:
model.load_weights(LogNormal_LSTM_path)

In [ ]:
def ve_tinh(model, evaluation_data, function=None):
    fig = plt.figure()
    i = 1
    score_df = pd.DataFrame({"Method": ["MAE", "RMSE", "R2"]})
    for name_set, train_set, test_set in evaluation_data:
        if function is None:
            y_pred = model.predict(train_set).flatten()
        else:
            pred = model.predict(train_set)
            n, p = tf.unstack(pred, num=2, axis=-1)
    
   
            mu = tf.expand_dims(n, -1)
            sigma = tf.expand_dims(p, -1)
            y_pred = function(mu, sigma).flatten()
            
        l = test_set[:,:,0].flatten()
        
        y_true = np.ma.compressed(np.ma.masked_where(l==0, l))
        y_pred = np.ma.compressed(np.ma.masked_where(l==0, y_pred))
        score_mae = "{0:.2f}".format(mean_absolute_error(y_true, y_pred))
        score_rmse = "{0:.2f}".format(sqrt(mean_squared_error(y_true, y_pred)))
        score_r2 = "{0:.3f}".format(r2_score(y_true, y_pred))
        score_df[name_set] = [score_mae, score_rmse, score_r2]
        ax = fig.add_subplot(6, 1, i)
        #ax.set_ylim(0, 100)
        ax.title.set_text(name_set)
        ax.title.set_fontsize(20)
        i += 1
        plt.plot(y_pred[0:3000])
        plt.plot(y_true[0:3000])
        ax = fig.add_subplot(6, 1, i)
        #ax.set_ylim(0, 100)
        i += 1
        plt.plot(y_pred[3000:6000])
        plt.plot(y_true[3000:6000])
        
    plt.subplots_adjust(hspace=0.45)
    fig.set_size_inches(15, i*2.2)
    return score_df.T

In [ ]:

from math import sqrt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import math
def mean_lognorm(mu, sig):
    return np.exp(mu+ 0.5*sig**2)

def mode_lognorm(mu, sig):
    return np.exp(mu - sig**2)

def median_lognorm(mu, sig):
    return np.exp(mu)

def var_lognorm(mu, sig):
    return np.exp(2*mu+sig**2)*(np.exp(sig**2)-1)

def pdf_lognorm(x, mu, sig):
    return 1/(x*sig*np.sqrt(2*math.pi))*np.exp(-0.5*((np.log(x) - mu)/sig)**2)

def inverse_cdf_weibull(x, alpha, beta):
    return beta*np.power((-np.log(1.-x)), 1./alpha)

def survival_weibull(x, alpha, beta):
    return np.e**-((x/beta)**alpha)

def confidence_lognormal(mu, sig):
    ms = np.exp(mu)
    sigs = np.exp(sig)
    mincon = ms/(sigs**2)
    maxcon = ms*(sigs**2)
    interval =maxcon - mincon
    return interval
    

In [ ]:
def rul_sys_seri2(nc,rulc,mu,sigma):
    rul_seri = min(rulc)
    t2 = rul_seri + 10
    t1 = max(0,rul_seri - 13)
    cdf_t2 = cdf_lognorm(t2, mu[:, isur ,0], sigma[:, isur ,0])
    cdf_t1 = cdf_lognorm(t1, mu[:, isur ,0], sigma[:, isur ,0])
    f = 1 - cdf_t2
    fail = np.zeros(nc)
    for i in np.arange(nc):
        p_i = cdf_t2[i] - cdf_t1[i]
        f[i] = 1
        fail[i] = p_i*np.prod(f)

    #print(rulc)
    #print(t1,t2)
    #print(fail)
    return np.sum(fail)

def rul_sys_seri(isur, tp, tn, rulc,mu,sigma):
    rul = min(rulc)
    t2 = rul + tp
    t1 = max(0,rul - tn)
    cdf_t2 = cdf_lognorm(t2, mu[:, isur ,0], sigma[:, isur ,0])
    cdf_t1 = cdf_lognorm(t1, mu[:, isur ,0], sigma[:, isur ,0])
    Rse_t1 = np.prod(1-cdf_t1)
    Rse_t2 = np.prod(1-cdf_t2)
    #print(rulc)
    #print(t1,t2)
    #print('Rse_t1',Rse_t1)
    #print('Rse_t2',Rse_t2)
    return (isur/(rul+isur), Rse_t1-Rse_t2)

def rul_sys_par(isur, tp, tn,rulc,mu,sigma):
    rul = max(rulc)
    t2 = rul + tp
    t1 = max(0,rul - tn)
    cdf_t2 = cdf_lognorm(t2, mu[:, isur ,0], sigma[:, isur ,0])
    cdf_t1 = cdf_lognorm(t1, mu[:, isur ,0], sigma[:, isur ,0])
    Rs_t1 = 1-np.prod(cdf_t1)
    Rs_t2 = 1-np.prod(cdf_t2)
    Rs = Rs_t1 - Rs_t2
    #print(rulc)
    #print(t1,t2)
    #print(Rs_t1, Rs_t2)
    return (isur/(rul+isur), Rs)

def rul_sys_parA(t1, t2,rul,mu,sigma):
    #rul = max(rulc)
   # t2 = rul + tp
    #t1 = max(0,rul - tn)
    cdf_t2 = cdf_lognorm(t2, mu, sigma)
    cdf_t1 = cdf_lognorm(t1, mu, sigma)
    Rs_t1 = 1-np.prod(cdf_t1)
    Rs_t2 = 1-np.prod(cdf_t2)
    Rs = Rs_t1 - Rs_t2
    #print(rulc)
    #print(t1,t2)
    #print(Rs_t1, Rs_t2)
    return Rs

def rul_par_ser(isur, ip,rulc,mu,sigma):
    ruls = min(np.append(max(rulc[:ip]), rulc[ip:]))
    #print(ruls)
    cdfA_t1 = rul_sys_parA(0, max(0,ruls-13), ruls,mu[:ip, isur ,0], sigma[:ip, isur ,0])
    cdfA_t2 = rul_sys_parA(0, ruls+10, ruls,mu[:ip, isur ,0], sigma[:ip, isur ,0])
    #print(cdfA_t1, cdfA_t2)

    cdf_t2 = np.append( cdfA_t2, cdf_lognorm(ruls+10, mu[ip:, isur ,0], sigma[ip:, isur ,0]))
    cdf_t1 =  np.append( cdfA_t1,cdf_lognorm(max(0,ruls-13), mu[ip:, isur ,0], sigma[ip:, isur ,0]))
    Rse_t1 = np.prod(1-cdf_t1)
    Rse_t2 = np.prod(1-cdf_t2)
    Rs = Rse_t1 - Rse_t2
    #print(cdf_t2)
    #print(cdf_t1)
    return (isur/(ruls+isur), Rs)

def rul_par_par(isur, ip,rulc,mu,sigma):
    ruls = min(np.append(max(rulc[:ip]), max(rulc[ip:])))
    cdfA_t1 = rul_sys_parA(0, max(0,ruls-13), ruls,mu[:ip, isur ,0], sigma[:ip, isur ,0])
    cdfA_t2 = rul_sys_parA(0, ruls+10, ruls,mu[:ip, isur ,0], sigma[:ip, isur ,0])
        #print(cdfA_t1, cdfA_t2)
    cdfB_t1 = rul_sys_parA(0, max(0,ruls-13), ruls,mu[ip:, isur ,0], sigma[ip:, isur ,0])
    cdfB_t2 = rul_sys_parA(0, ruls+10, ruls,mu[ip:, isur ,0], sigma[ip:, isur ,0])

    cdf_t2 = np.append( cdfA_t2, cdfB_t2)
    cdf_t1 =  np.append( cdfA_t1, cdfB_t1)
    Rse_t1 = np.prod(1-cdf_t1)
    Rse_t2 = np.prod(1-cdf_t2)
    #print(Rse_t1, Rse_t2)
    Rs = Rse_t1 - Rse_t2
    #print('thoi gian theo doi:',isur/(isur+ruls))
    #print('rul sys par-serie:',Rs)
    return (isur/(ruls+isur), Rs)

In [ ]:


nc = 4
pc = .9 #% thoi gian theo doi
tp = 10
tn = 13
engine_ids = np.random.choice(np.arange(100), nc, replace=False)
#engine_ids = np.array([70, 86, 87, 97])-70
#engine_ids = np.array([88, 127, 61, 40])-70
engine_ids = np.sort(engine_ids)
X_sys = X[engine_ids, :,:]
Y_sys = Y[engine_ids, :,:]

maxlife=[]

for i in np.arange(nc):
    tt = np.where(Y_sys[i,:,0]==0)
    if len(tt[0])!=0:
        maxlife=np.append(maxlife,np.min(np.where(Y_sys[i,:,0]==0)))
    else:
        maxlife=np.append(maxlife,362)
tsur = min(maxlife) #thoi gian theo doi cua ca 4 engines

isur = np.int(np.round(tsur*pc))-1
print('engine id:', engine_ids+70)
print('thoi gian theo doi cua ca 4 engines',isur)
X_sur = np.zeros(np.shape(X_sys))
Y_sur = np.zeros(np.shape(Y_sys))
X_sur[:,:isur+1,:] = X_sys[:,:isur+1,:]
Y_sur[:,:isur+1,:] = Y_sys[:,:isur+1,:]



pred = model.predict(X_sur)
n, p = tf.unstack(pred, num=2, axis=-1)

 # Add one dimension to make the right shape
mu = tf.expand_dims(n, -1)
sigma = tf.expand_dims(p, -1)

ip = 2
rulc = Y_sys[:,isur,0]
print(rulc)

ruls = min(np.append(max(rulc[:ip]), max(rulc[ip:])))
print(ruls)
cdfA_t1 = rul_sys_parA(0, max(0,ruls-13), ruls,mu[:ip, isur ,0], sigma[:ip, isur ,0])
cdfA_t2 = rul_sys_parA(0, ruls+10, ruls,mu[:ip, isur ,0], sigma[:ip, isur ,0])
    #print(cdfA_t1, cdfA_t2)
cdfB_t1 = rul_sys_parA(0, max(0,ruls-13), ruls,mu[ip:, isur ,0], sigma[ip:, isur ,0])
cdfB_t2 = rul_sys_parA(0, ruls+10, ruls,mu[ip:, isur ,0], sigma[ip:, isur ,0])
           
cdf_t2 = np.append( cdfA_t2, cdfB_t2)
cdf_t1 =  np.append( cdfA_t1, cdfB_t1)
Rse_t1 = np.prod(1-cdf_t1)
Rse_t2 = np.prod(1-cdf_t2)
print(Rse_t1, Rse_t2)
Rs = Rse_t1 - Rse_t2
print('thoi gian theo doi:',isur/(isur+ruls))
print('rul sys par-serie:',Rs)

print('rul sys par2+serie:',rul_par_ser(isur,2,rulc,mu,sigma))
print('rul sys par3+serie:',rul_par_ser(isur,3,rulc,mu,sigma))
print('rul sys serie:', rul_sys_seri(isur,tp, tn, rulc,mu,sigma))
print('rul sys par:',rul_sys_par(isur, tp, tn,rulc,mu,sigma))

In [ ]:
nc = 4
pc = 1 #% thoi gian theo doi
tp = 10
tn = 13
engine_ids = np.random.choice(np.arange(100), nc, replace=False)
#engine_ids = np.array([70, 86, 87, 97])-70
#engine_ids = np.array([78, 84, 89, 97])-70
engine_ids = np.sort(engine_ids)
X_sys = test_X[engine_ids, :,:]
Y_sys = test_Y[engine_ids, :,:]

maxlife=[]

for i in np.arange(nc):
    tt = np.where(Y_sys[i,:,0]==0)
    if len(tt[0])!=0:
        maxlife=np.append(maxlife,np.min(np.where(Y_sys[i,:,0]==0)))
    else:
        maxlife=np.append(maxlife,362)
tsur = min(maxlife) #thoi gian theo doi cua ca 4 engines

isur = np.int(np.round(tsur*pc))-1
print('engine id:', engine_ids)
print('thoi gian theo doi cua ca 4 engines',isur)
X_sur = np.zeros(np.shape(X_sys))
Y_sur = np.zeros(np.shape(Y_sys))
X_sur[:,:isur+1,:] = X_sys[:,:isur+1,:]
Y_sur[:,:isur+1,:] = Y_sys[:,:isur+1,:]



pred = model.predict(X_sur)
n, p = tf.unstack(pred, num=2, axis=-1)

 # Add one dimension to make the right shape
mu = tf.expand_dims(n, -1)
sigma = tf.expand_dims(p, -1)

#ip = 2
rulc = Y_sys[:,isur,0]
print(rulc)

print('rul sys par2+serie:',rul_par_ser(isur,2,rulc,mu,sigma))
print('rul sys par3+serie:',rul_par_ser(isur,3,rulc,mu,sigma))
print('rul sys serie:', rul_sys_seri(isur,tp, tn, rulc,mu,sigma))
print('rul sys par:',rul_sys_par(isur,tp, tn,rulc,mu,sigma))

In [ ]:
#pc = .9 #% thoi gian theo doi



def rul_sys_seri2(pc, tp, tn, rulc,mu,sigma):
    rul = min(rulc)
    isur = np.int(np.round(rul*pc))
    t2 = rul + tp
    t1 = max(0,rul - tn)
    cdf_t2 = cdf_lognorm(t2, mu[:, isur ,0], sigma[:, isur ,0])
    cdf_t1 = cdf_lognorm(t1, mu[:, isur ,0], sigma[:, isur ,0])
    Rse_t1 = np.prod(1-cdf_t1)
    Rse_t2 = np.prod(1-cdf_t2)
    #print(rulc)
    #print(t1,t2)
    #print('Rse_t1',Rse_t1)
    #print('Rse_t2',Rse_t2)
    return (isur/(rul+isur), Rse_t1-Rse_t2)

def rul_sys_par2(pc, tp, tn,rulc,mu,sigma):
    rul = max(rulc)
    isur = np.int(np.round(rul*pc))
    t2 = rul + tp
    t1 = max(0,rul - tn)
    cdf_t2 = cdf_lognorm(t2, mu[:, isur ,0], sigma[:, isur ,0])
    cdf_t1 = cdf_lognorm(t1, mu[:, isur ,0], sigma[:, isur ,0])
    Rs_t1 = 1-np.prod(cdf_t1)
    Rs_t2 = 1-np.prod(cdf_t2)
    Rs = Rs_t1 - Rs_t2
    #print(rulc)
    #print(t1,t2)
    #print(Rs_t1, Rs_t2)
    return (isur/(rul+isur), Rs)

def rul_sys_parA2(t1, t2,rul,mu,sigma, rulc):
    #rul = max(rulc)
   # t2 = rul + tp
    #t1 = max(0,rul - tn)
    
    cdf_t2 = cdf_lognorm(t2, mu, sigma)
    cdf_t1 = cdf_lognorm(t1, mu, sigma)
    
    if (len(np.where(rulc==0)[0]) > 0):
        cdf_t1 = np.where(rulc==0, 1, cdf_t1)
        cdf_t2 = np.where(rulc==0, 1, cdf_t2)
    
    Rs_t1 = 1-np.prod(cdf_t1)
    Rs_t2 = 1-np.prod(cdf_t2)
    Rs = Rs_t1 - Rs_t2
    #print(rulc)
    #print(t1,t2)
    #print(Rs_t1, Rs_t2)
    return Rs

def rul_par_ser2(pc, ip,rulc,mu,sigma):
    ruls = min(np.append(max(rulc[:ip]), rulc[ip:]))
    isur = np.int(np.round(ruls*pc))
    #print(ruls)
    cdfA_t1 = rul_sys_parA(0, max(0,ruls-13), ruls,mu[:ip, isur ,0], sigma[:ip, isur ,0])
    cdfA_t2 = rul_sys_parA(0, ruls+10, ruls,mu[:ip, isur ,0], sigma[:ip, isur ,0])
    #print(cdfA_t1, cdfA_t2)

    cdf_t2 = np.append( cdfA_t2, cdf_lognorm(ruls+10, mu[ip:, isur ,0], sigma[ip:, isur ,0]))
    cdf_t1 =  np.append( cdfA_t1,cdf_lognorm(max(0,ruls-13), mu[ip:, isur ,0], sigma[ip:, isur ,0]))
    Rse_t1 = np.prod(1-cdf_t1)
    Rse_t2 = np.prod(1-cdf_t2)
    Rs = Rse_t1 - Rse_t2
    #print(cdf_t2)
    #print(cdf_t1)
    return (isur/(ruls+isur), Rs)

def rul_par_par2(pc, ip,rulc,mu,sigma):
    ruls = min(np.append(max(rulc[:ip]), max(rulc[ip:])))
    isur = np.int(np.round(ruls*pc))
    cdfA_t1 = rul_sys_parA(0, max(0,ruls-13), ruls,mu[:ip, isur ,0], sigma[:ip, isur ,0])
    cdfA_t2 = rul_sys_parA(0, ruls+10, ruls,mu[:ip, isur ,0], sigma[:ip, isur ,0])
        #print(cdfA_t1, cdfA_t2)
    cdfB_t1 = rul_sys_parA(0, max(0,ruls-13), ruls,mu[ip:, isur ,0], sigma[ip:, isur ,0])
    cdfB_t2 = rul_sys_parA(0, ruls+10, ruls,mu[ip:, isur ,0], sigma[ip:, isur ,0])

    cdf_t2 = np.append( cdfA_t2, cdfB_t2)
    cdf_t1 =  np.append( cdfA_t1, cdfB_t1)
    Rse_t1 = np.prod(1-cdf_t1)
    Rse_t2 = np.prod(1-cdf_t2)
    #print(Rse_t1, Rse_t2)
    Rs = Rse_t1 - Rse_t2
    #print('thoi gian theo doi:',isur/(isur+ruls))
    #print('rul sys par-serie:',Rs)
    return (isur/(ruls+isur), Rs)

def test_sys(pc,num,XX,YY):
    nc = 4

    tp = 10
    tn = 13
    rpss2=[]
    rpss3=[]
    rpars=[]
    rseris=[]
    tsurr=[]
    tss2 = []
    tss3 = []
    tpars = []
    tseris = []
    for i in np.arange(num):
        #engine_ids = np.random.choice(np.arange(100), nc, replace=False)
        engine_ids = np.random.choice(np.arange(30), nc, replace=False)
        ##engine_ids = np.array([78, 79, 89, 97])-70
        #engine_ids = np.array([78, 84, 89, 97])-70
        engine_ids = np.sort(engine_ids)
        #X_sys = val_X[engine_ids, :,:]
        #Y_sys = val_Y[engine_ids, :,:]
        X_sys = XX[engine_ids, :,:]
        Y_sys = YY[engine_ids, :,:]

        maxlife=[]

        for i in np.arange(nc):
            tt = np.where(Y_sys[i,:,0]==0)
            if len(tt[0])!=0:
                maxlife=np.append(maxlife,np.min(np.where(Y_sys[i,:,0]==0)))
            else:
                maxlife=np.append(maxlife,362)
        tsur = min(maxlife) #thoi gian theo doi cua ca 4 engines

        isur = np.int(np.floor(tsur*pc))+1#-1 #- 2
        #print('engine id:', engine_ids)
        #print('thoi gian theo doi cua ca 4 engines',isur)
        X_sur = np.zeros(np.shape(X_sys))
        Y_sur = np.zeros(np.shape(Y_sys))
        X_sur[:,:isur+1,:] = X_sys[:,:isur+1,:]
        Y_sur[:,:isur+1,:] = Y_sys[:,:isur+1,:]



        pred = model.predict(X_sur)
        n, p = tf.unstack(pred, num=2, axis=-1)

         # Add one dimension to make the right shape
        mu = tf.expand_dims(n, -1)
        sigma = tf.expand_dims(p, -1)

        #ip = 2
        rulc = Y_sys[:,isur,0]

        ts2, rps2 = rul_par_ser(isur,2,rulc,mu,sigma)
        #ts3, rps3 = rul_par_ser(isur,3,rulc,mu,sigma)
        ts3, rps3 = rul_par_par(isur,2,rulc,mu,sigma)
        tpar, rpar = rul_sys_par(isur, tp, tn,rulc,mu,sigma)
        tseri, rseri = rul_sys_seri(isur, tp, tn,rulc,mu,sigma)
        
        rpss2 = np.append(rpss2,rps2)
        rpss3 = np.append(rpss3,rps3)
        rpars = np.append(rpars,rpar)
        rseris = np.append(rseris, rseri)
        
        tss2 = np.append(tss2,ts2)
        tss3 = np.append(tss3,ts3)
        tpars = np.append(tpars,tpar)
        tseris = np.append(tseris, tseri)
        
        tsurr = np.append(tsurr,isur)
        
    #print('tsur: ', tsurr)
    #print('rps2: ', rps2)
    #print('rps3: ', rps3)
    #print('rpar: ', rpar)
    #print('rseri: ', rseri)
                      
    return (np.mean(tsurr),np.mean(rpss2), np.mean(tss2), np.mean(rpss3), np.mean(tss3),
            np.mean(rpars), np.mean(tpars),  np.mean(rseris), np.mean(tseris))


In [ ]:
#pc = .9 #% thoi gian theo doi


def test_sys3(pc,num,XX,YY):
    nc = 1

    tp = 10
    tn = 13
    rpss2=[]
    rpss3=[]
    rpars=[]
    rseris=[]
    tsurr=[]
    tss2 = []
    tss3 = []
    tpars = []
    tseris = []
    for i in np.arange(num):
        #engine_ids = np.random.choice(np.arange(100), nc, replace=False)
        engine_ids = np.random.choice(np.arange(30), nc, replace=False)
        ##engine_ids = np.array([78, 79, 89, 97])-70
        #engine_ids = np.array([78, 84, 89, 97])-70
        engine_ids = np.sort(engine_ids)
        #X_sys = val_X[engine_ids, :,:]
        #Y_sys = val_Y[engine_ids, :,:]
        X_sys = XX[engine_ids, :,:]
        Y_sys = YY[engine_ids, :,:]

        maxlife=[]

        for i in np.arange(nc):
            tt = np.where(Y_sys[i,:,0]==0)
            if len(tt[0])!=0:
                maxlife=np.append(maxlife,np.min(np.where(Y_sys[i,:,0]==0)))
            else:
                maxlife=np.append(maxlife,362)
        
        ###############
        #tinh sys-par
        
        tsur = max(maxlife) #thoi gian theo doi cua ca 4 engines
        isur = np.int(np.floor(tsur*pc))+1
        #isur = np.int(tsur - 3)
        #print('engine id:', engine_ids)
        #print('thoi gian theo doi cua ca 4 engines',isur)
        X_sur = np.zeros(np.shape(X_sys))
        Y_sur = np.zeros(np.shape(Y_sys))
        X_sur[:,:isur+1,:] = X_sys[:,:isur+1,:]
        Y_sur[:,:isur+1,:] = Y_sys[:,:isur+1,:]
        pred = model.predict(X_sur)
        n, p = tf.unstack(pred, num=2, axis=-1)

         # Add one dimension to make the right shape
        mu = tf.expand_dims(n, -1)
        sigma = tf.expand_dims(p, -1)

        #ip = 2
        rulc = Y_sys[:,isur,0]
        #tpar, rpar = rul_sys_par(isur,tp, tn,rulc,mu,sigma)
        
        
        
        rul = max(rulc)
        t2 = rul + tp
        t1 = max(0,rul - tn)
        cdf_t2 = cdf_lognorm(t2, mu[:, isur ,0], sigma[:, isur ,0])
        cdf_t1 = cdf_lognorm(t1, mu[:, isur ,0], sigma[:, isur ,0])
        
        if (len(np.where(rulc==0)[0]) > 0):
            cdf_t1 = np.where(rulc==0, 1, cdf_t1)
            cdf_t2 = np.where(rulc==0, 1, cdf_t2)
        
        Rs_t1 = 1-np.prod(cdf_t1)
        Rs_t2 = 1-np.prod(cdf_t2)
        rpar = Rs_t1 - Rs_t2
        tpar = isur/(rul+isur)
        rpars = np.append(rpars,rpar)
        tpars = np.append(tpars,tpar)
        
         
    

     ###############
        #tinh serie
        lifeserie = min(maxlife) #thoi gian theo doi cua ca 4 engines
        isur = np.int(np.floor(lifeserie*pc))+1#-1 #- 2
        #isur = np.int(lifeserie - 3)
        X_sur = np.zeros(np.shape(X_sys))
        Y_sur = np.zeros(np.shape(Y_sys))
        X_sur[:,:isur+1,:] = X_sys[:,:isur+1,:]
        Y_sur[:,:isur+1,:] = Y_sys[:,:isur+1,:]
        pred = model.predict(X_sur)
        n, p = tf.unstack(pred, num=2, axis=-1)

         # Add one dimension to make the right shape
        mu = tf.expand_dims(n, -1)
        sigma = tf.expand_dims(p, -1)
        rulc = Y_sys[:,isur,0]
        tseri, rseri = rul_sys_seri(isur,tp, tn, rulc,mu,sigma)
        rseris = np.append(rseris, rseri)
        tseris = np.append(tseris, tseri)
        #####
        
        
        
        
        
        
        tsurr = np.append(tsurr,isur)
        
    #print('tsur: ', tsurr)
    #print('rps2: ', rps2)
    #print('rps3: ', rps3)
    #print('rpar: ', rpar)
    #print('rseri: ', rseri)
                      
    return (np.mean(tsurr),np.mean(rpss2), np.mean(tss2), np.mean(rpss3), np.mean(tss3),
            np.mean(rpars), np.mean(tpars),  np.mean(rseris), np.mean(tseris))


In [ ]:
#pc = .9 #% thoi gian theo doi


def test_sys2(pc,num,XX,YY):
    nc = 4

    tp = 10
    tn = 13
    rpss2=[]
    rpss3=[]
    rpars=[]
    rseris=[]
    tsurr=[]
    tss2 = []
    tss3 = []
    tpars = []
    tseris = []
    for i in np.arange(num):
        #engine_ids = np.random.choice(np.arange(100), nc, replace=False)
        engine_ids = np.random.choice(np.arange(30), nc, replace=False)
        ##engine_ids = np.array([78, 79, 89, 97])-70
        #engine_ids = np.array([78, 84, 89, 97])-70
        engine_ids = np.sort(engine_ids)
        #X_sys = val_X[engine_ids, :,:]
        #Y_sys = val_Y[engine_ids, :,:]
        X_sys = XX[engine_ids, :,:]
        Y_sys = YY[engine_ids, :,:]

        maxlife=[]

        for i in np.arange(nc):
            tt = np.where(Y_sys[i,:,0]==0)
            if len(tt[0])!=0:
                maxlife=np.append(maxlife,np.min(np.where(Y_sys[i,:,0]==0)))
            else:
                maxlife=np.append(maxlife,362)
        
        ###############
        #tinh sys-par
        
        tsur = max(maxlife) #thoi gian theo doi cua ca 4 engines

        isur = np.int(np.floor(tsur*pc))+2#-1 #- 2
        #isur = np.int(np.floor(tsur*pc))+2#-1 #- 2
        #print('engine id:', engine_ids)
        #print('thoi gian theo doi cua ca 4 engines',isur)
        X_sur = np.zeros(np.shape(X_sys))
        Y_sur = np.zeros(np.shape(Y_sys))
        X_sur[:,:isur+1,:] = X_sys[:,:isur+1,:]
        Y_sur[:,:isur+1,:] = Y_sys[:,:isur+1,:]
        pred = model.predict(X_sur)
        n, p = tf.unstack(pred, num=2, axis=-1)

         # Add one dimension to make the right shape
        mu = tf.expand_dims(n, -1)
        sigma = tf.expand_dims(p, -1)

        #ip = 2
        rulc = Y_sys[:,isur,0]
        #tpar, rpar = rul_sys_par(isur,tp, tn,rulc,mu,sigma)
        
        
        
        rul = max(rulc)
        t2 = rul + tp
        t1 = max(0,rul - tn)
        cdf_t2 = cdf_lognorm(t2, mu[:, isur ,0], sigma[:, isur ,0])
        cdf_t1 = cdf_lognorm(t1, mu[:, isur ,0], sigma[:, isur ,0])
        
        if (len(np.where(rulc==0)[0]) > 0):
            cdf_t1 = np.where(rulc==0, 1, cdf_t1)
            cdf_t2 = np.where(rulc==0, 1, cdf_t2)
        
        Rs_t1 = 1-np.prod(cdf_t1)
        Rs_t2 = 1-np.prod(cdf_t2)
        rpar = Rs_t1 - Rs_t2
        #tpar = isur/(rul+isur)
        tpar = rul #- isur
        rpars = np.append(rpars,rpar)
        tpars = np.append(tpars,tpar)
        
         ###############
        #tinh combined1
        #ts2, rps2 = rul_par_ser(isur,2,rulc,mu,sigma)
        ip = 2
        lifesys1 = min(np.append(max(maxlife[:2]), maxlife[2:]))
        isur = np.int(np.floor(lifesys1*pc))#+1#-1 #- 2
        #print('engine id:', engine_ids)
        #print('thoi gian theo doi cua ca 4 engines',isur)
        X_sur = np.zeros(np.shape(X_sys))
        Y_sur = np.zeros(np.shape(Y_sys))
        X_sur[:,:isur+1,:] = X_sys[:,:isur+1,:]
        Y_sur[:,:isur+1,:] = Y_sys[:,:isur+1,:]
        pred = model.predict(X_sur)
        n, p = tf.unstack(pred, num=2, axis=-1)

         # Add one dimension to make the right shape
        mu = tf.expand_dims(n, -1)
        sigma = tf.expand_dims(p, -1)

        
        rulc = Y_sys[:,isur,0]
        ruls = min(np.append(max(rulc[:ip]), rulc[ip:]))
    
    #print(ruls)
        cdfA_t1 = rul_sys_parA2(0, max(0,ruls-13), ruls,mu[:ip, isur ,0], sigma[:ip, isur ,0],rulc[:ip])
        cdfA_t2 = rul_sys_parA2(0, ruls+10, ruls,mu[:ip, isur ,0], sigma[:ip, isur ,0],rulc[:ip])
        
        
    #print(cdfA_t1, cdfA_t2)
        cdf_t2 = np.append( cdfA_t2, cdf_lognorm(ruls+10, mu[ip:, isur ,0], sigma[ip:, isur ,0]))
        cdf_t1 =  np.append( cdfA_t1,cdf_lognorm(max(0,ruls-13), mu[ip:, isur ,0], sigma[ip:, isur ,0]))
        
        if (len(np.where(rulc[ip:]==0)[0]) > 0):
            ind_pos = np.where(rulc[ip:]==0)[0] - 1
            for ihh in ind_pos:
                cdf_t1[ihh] = 1
                cdf_t2[ihh] = 1
        
        Rse_t1 = np.prod(1-cdf_t1)
        Rse_t2 = np.prod(1-cdf_t2)
        rps2 = Rse_t1 - Rse_t2
        #ts2 = isur/(ruls+isur)
        ts2 = ruls #- isur
        rpss2 = np.append(rpss2,rps2)
        tss2 = np.append(tss2,ts2)
        
         ###############
        #tinh combined2
        #ts3, rps3 = rul_par_par(isur, ip,rulc,mu,sigma)
        lifesys1 = min(np.append(max(maxlife[:ip]), max(maxlife[ip:])))
        #isur = np.int(np.round(lifesys1*pc))
        isur = np.int(np.floor(lifesys1*pc)) -2 #+ 1
        X_sur = np.zeros(np.shape(X_sys))
        Y_sur = np.zeros(np.shape(Y_sys))
        X_sur[:,:isur+1,:] = X_sys[:,:isur+1,:]
        Y_sur[:,:isur+1,:] = Y_sys[:,:isur+1,:]
        pred = model.predict(X_sur)
        n, p = tf.unstack(pred, num=2, axis=-1)

         # Add one dimension to make the right shape
        mu = tf.expand_dims(n, -1)
        sigma = tf.expand_dims(p, -1)
        rulc = Y_sys[:,isur,0]
        
        ruls = min(np.append(max(rulc[:ip]), max(rulc[ip:])))
        
        cdfA_t1 = rul_sys_parA2(0, max(0,ruls-13), ruls,mu[:ip, isur ,0], sigma[:ip, isur ,0],rulc[:ip])
        cdfA_t2 = rul_sys_parA2(0, ruls+10, ruls,mu[:ip, isur ,0], sigma[:ip, isur ,0],rulc[:ip])
        #print(cdfA_t1, cdfA_t2)
        cdfB_t1 = rul_sys_parA2(0, max(0,ruls-13), ruls,mu[ip:, isur ,0], sigma[ip:, isur ,0],rulc[ip:])
        cdfB_t2 = rul_sys_parA2(0, ruls+10, ruls,mu[ip:, isur ,0], sigma[ip:, isur ,0],rulc[ip:])

        cdf_t2 = np.append( cdfA_t2, cdfB_t2)
        cdf_t1 =  np.append( cdfA_t1, cdfB_t1)
        Rse_t1 = np.prod(1-cdf_t1)
        Rse_t2 = np.prod(1-cdf_t2)
    #print(Rse_t1, Rse_t2)
        rps3 = Rse_t1 - Rse_t2
        #ts3 = isur/(ruls+isur)
        ts3 = ruls #- isur
        rpss3 = np.append(rpss3,rps3)
        tss3 = np.append(tss3,ts3)
    #print('thoi gian theo doi:',isur/(isur+ruls))
    #print('rul sys par-serie:',Rs)
    

     ###############
        #tinh serie
        lifeserie = min(maxlife) #thoi gian theo doi cua ca 4 engines
        isur = np.int(np.floor(lifeserie*pc))+1#-1 #- 2
        X_sur = np.zeros(np.shape(X_sys))
        Y_sur = np.zeros(np.shape(Y_sys))
        X_sur[:,:isur+1,:] = X_sys[:,:isur+1,:]
        Y_sur[:,:isur+1,:] = Y_sys[:,:isur+1,:]
        pred = model.predict(X_sur)
        n, p = tf.unstack(pred, num=2, axis=-1)

         # Add one dimension to make the right shape
        mu = tf.expand_dims(n, -1)
        sigma = tf.expand_dims(p, -1)
        rulc = Y_sys[:,isur,0]
        tseri, rseri = rul_sys_seri(isur,tp, tn, rulc,mu,sigma)
        ruls = min(rulc)
        rseris = np.append(rseris, rseri)
        tseris = np.append(tseris, tseri)
        #####
        
        
        
        
        
        #tsurr: horizon prognostic 
        tsurr = np.append(tsurr,ruls)
        
    #print('tsur: ', tsurr)
    #print('rps2: ', rps2)
    #print('rps3: ', rps3)
    #print('rpar: ', rpar)
    #print('rseri: ', rseri)
                      
    return (np.mean(tsurr),np.mean(rpss2), np.mean(tss2), np.mean(rpss3), np.mean(tss3),
            np.mean(rpars), np.mean(tpars),  np.mean(rseris), np.mean(tseris))


In [ ]:
print("Median")
ve_tinh(model,
                  [("Train", train_X, train_Y), 
                   ("Validation", val_X, val_Y), 
                   ("Test", test_X, test_Y)],
                 function = median_lognorm)


In [ ]:
Metric_all_eval(median_lognorm)

In [ ]:
Metric_all_eval(mode_lognorm) #dung 70 cai dau train, 30 cai sau val

In [ ]:
Metric_all_eval(mean_lognorm) #dung 70 cai dau train, 30 cai sau val

In [ ]:
train_set = test_X
test_set = test_Y
function = confidence_lognormal
name_set = "Test"
pred = model.predict(train_set)
n, p = tf.unstack(pred, num=2, axis=-1)

 # Add one dimension to make the right shape
mu = tf.expand_dims(n, -1)
sigma = tf.expand_dims(p, -1)
y_pred = mean_lognorm(mu, sigma).flatten()

l = test_set[:,:,0].flatten()
        # To validate we remove the right padding
y_true = np.ma.compressed(np.ma.masked_where(l==0, l))
y_pred = np.ma.compressed(np.ma.masked_where(l==0, y_pred))

kk = np.diff(test_df0['RUL'].values)
ii = np.where(kk > 0)[0]
index = np.append(ii, 13095)
    
ms = np.exp(mu).flatten()
sigs = np.exp(sigma).flatten()
ms = np.ma.compressed(np.ma.masked_where(l==0, ms))[index]
sigs = np.ma.compressed(np.ma.masked_where(l==0, sigs))[index]
    
mincon = ms/(sigs**2)
maxcon = ms*(sigs**2)
interval =maxcon - mincon
    
ytrue = y_true[index]

be = np.zeros(len(mincon))
be_130 = np.zeros(len(mincon))
for ih in np.arange(len(mincon)):
    if (truth[ih] <= maxcon[ih]) & (truth[ih] >= mincon[ih]):
        be[ih] = 1
    if (ytrue[ih] <= maxcon[ih]) & (truth[ih] >= mincon[ih]):
        be_130[ih] = 1
interval = maxcon-mincon
NMPIW = sum(interval)/(len(mincon)*(max(truth)-min(truth)))
NMPIW_130 = sum(interval)/(len(mincon)*(max(ytrue)-min(ytrue)))
print(be)
print(be_130)
print(sum(be))
print(sum(be_130))
print(NMPIW)   
print(NMPIW_130)
plt.figure(figsize=(16, 8))

# only one line may be specified; full height

# multiple lines with varying ymin and ymax
rangx = np.arange(100)+1
plt.vlines(x=rangx , ymin=mincon, ymax=maxcon, colors='blue', ls='-', lw=1.5, label='95% prediction interval')
plt.scatter(rangx, truth, s = 50, c = 'red', marker = 'x', label='Ground truth RUL')
plt.scatter(rangx, y_pred[index], s = 30, c = 'black', marker = 'o', label='Median predicted RUL')
# place legend outside
#plt.legend(bbox_to_anchor=(1.0, 1), loc='upper left')
plt.xlabel('Test unit number', fontsize=16)
plt.ylabel('RUL (cycle)', fontsize=16)
plt.legend(loc='lower left', prop={'size': 12})

plt.savefig('Images_paper/LSTM_lognorm/Prediction_results_test_2.pdf')

plt.show()



In [ ]:
#y_69 = model.predict(X)
n, p = tf.unstack(y_69[68,:,:], num=2, axis=-1)
pmax=.96
pmin= .4
mu = tf.expand_dims(n, -1)
sigma = tf.expand_dims(p, -1)
pred_69 = median_lognorm(mu, sigma).flatten()
RUL_69 = train_df[train_df['id']==69]['RUL'].values
t = np.arange(len(RUL_69)) + 1
imax = np.int(np.round(len(RUL_69)*pmax))
imin = np.int(np.round(len(RUL_69)*pmin))
plt.figure(figsize=(8, 4))

ms = np.exp(mu).flatten()
sigs = np.exp(sigma).flatten()

    
mincon = ms/(sigs**2)
maxcon = ms*(sigs**2)
# only one line may be specified; full height

# multiple lines with varying ymin and ymax

# place legend outside
#plt.legend(bbox_to_anchor=(1.0, 1), loc='upper left')
plt.plot(t[imin:imax],pred_69[imin:imax],'b',label='Median predicted RUL')
plt.plot(t[imin:imax],RUL_69[imin:imax],'r',label='Piece-wise linear truth RUL')
plt.fill_between(t[imin:imax], mincon[imin:imax], maxcon[imin:imax], color='b', alpha=.2,label='95% prediction interval')
plt.xlabel('Life cycles', fontsize=16)
plt.ylabel('RUL (cycle)', fontsize=16)
plt.legend(loc='lower left', prop={'size': 12})

plt.savefig('Images_paper/Prediction_train_unit69.pdf')

plt.show()

In [ ]:
y_69 = model.predict(X)
n, p = tf.unstack(y_69[68,:,:], num=2, axis=-1)
pmax=.96
pmin= .4
mu = tf.expand_dims(n, -1)
sigma = tf.expand_dims(p, -1)
pred_69 = median_lognorm(mu, sigma).flatten()
RUL_69 = train_df[train_df['id']==69]['RUL'].values
t = np.arange(len(RUL_69)) + 1
imax = np.int(np.round(len(RUL_69)*pmax))
imin = np.int(np.round(len(RUL_69)*pmin))
plt.figure(figsize=(8, 4))

ms = np.exp(mu).flatten()
sigs = np.exp(sigma).flatten()

    
mincon = ms/(sigs**2)
maxcon = ms*(sigs**2)
# only one line may be specified; full height

# multiple lines with varying ymin and ymax

# place legend outside
#plt.legend(bbox_to_anchor=(1.0, 1), loc='upper left')
plt.plot(t[imin:imax],pred_69[imin:imax],'b',label='Median predicted RUL')
plt.plot(t[imin:imax],RUL_69[imin:imax],'r',label='Piece-wise linear truth RUL')
plt.fill_between(t[imin:imax], mincon[imin:imax], maxcon[imin:imax], color='b', alpha=.2,label='95% prediction interval')
plt.xlabel('Life cycles', fontsize=16)
plt.ylabel('RUL (cycle)', fontsize=16)
plt.legend(loc='lower left', prop={'size': 12})

plt.savefig('Images_paper/Prediction_train_unit69_2.pdf')

plt.show()

In [ ]:
n, p = tf.unstack(y_69[4,:,:], num=2, axis=-1)
pmax=.95
pmin= 0.4
mu = tf.expand_dims(n, -1)
sigma = tf.expand_dims(p, -1)
pred_69 = median_lognorm(mu, sigma).flatten()
RUL_69 = train_df[train_df['id']==5]['RUL'].values
t = np.arange(len(RUL_69)) + 1
imax = np.int(np.round(len(RUL_69)*pmax))
imin = np.int(np.round(len(RUL_69)*pmin))
plt.figure(figsize=(8, 4))

ms = np.exp(mu).flatten()
sigs = np.exp(sigma).flatten()

    
mincon = ms/(sigs**2)
maxcon = ms*(sigs**2)

# place legend outside
#plt.legend(bbox_to_anchor=(1.0, 1), loc='upper left')
plt.plot(t[imin:imax],pred_69[imin:imax],'b',label='Median predicted RUL')
plt.plot(t[imin:imax],RUL_69[imin:imax],'r',label='Piece-wise linear truth RUL')
plt.fill_between(t[imin:imax], mincon[imin:imax], maxcon[imin:imax], color='b', alpha=.2,label='95% prediction interval')
plt.xlabel('Life cycles', fontsize=16)
plt.ylabel('RUL (cycle)', fontsize=16)
plt.legend(loc='lower left', prop={'size': 12})

plt.savefig('Images_paper/Prediction_train_unit5.pdf')

plt.show()

In [ ]:
pred = model.predict(test_X)

In [ ]:
u=35
n, p = tf.unstack(pred[u-1,:,:], num=2, axis=-1)
pmax=1
pmin= 0.35
mu = tf.expand_dims(n, -1)
sigma = tf.expand_dims(p, -1)
pred_69 = median_lognorm(mu, sigma).flatten()
RUL_69 = test_df[test_df['id']==u]['RUL'].values
RUL = test_df0[test_df0['id']==u]['RUL'].values
t = np.arange(len(RUL_69)) + 1
imax = np.int(np.round(len(RUL_69)*pmax))
imin = np.int(np.round(len(RUL_69)*pmin))
plt.figure(figsize=(8, 4))

ms = np.exp(mu).flatten()
sigs = np.exp(sigma).flatten()

    
mincon = ms/(sigs**2)
maxcon = ms*(sigs**2)

# place legend outside
#plt.legend(bbox_to_anchor=(1.0, 1), loc='upper left')
plt.plot(t[imin:imax],pred_69[imin:imax],'b',label='Median predicted RUL')
plt.plot(t[imin:imax],RUL_69[imin:imax],'r',label='Piece-wise linear truth RUL')
plt.plot(t[imin:imax],RUL[imin:imax],'r--',label='Ground truth RUL')
plt.fill_between(t[imin:imax], mincon[imin:imax], maxcon[imin:imax], color='b', alpha=.2,label='95% prediction interval')
plt.xlabel('Life cycles', fontsize=16)
plt.ylabel('RUL (cycle)', fontsize=16)
plt.legend(loc='upper right', prop={'size': 12})

#plt.savefig('Images_paper/LSTM_lognorm/Prediction_test_unit35_2.pdf')

plt.show()

In [ ]:
u=35
xx = np.arange(5,24,0.1)
pdf = pdf_lognorm(xx,mu[len(RUL)-1],sigma[len(RUL)-1])

plt.figure(figsize=(7, 4))
plt.plot(xx, pdf, label = 'Predicted RUL pdf')
plt.vlines(x=median_lognorm(mu[len(RUL)-1],sigma[len(RUL)-1]), ymin=0, ymax=pdf_lognorm(median_lognorm(mu[len(RUL)-1],sigma[len(RUL)-1]),mu[len(RUL)-1],sigma[len(RUL)-1]), colors='blue', lw=2, ls=':', label='Median predicted RUL')
plt.scatter(truth[u-1],0,s = 50, c = 'red', marker = 'x', label='Ground truth RUL')
xfill = np.arange(mincon[len(RUL)-1],maxcon[len(RUL)-1],0.1)
#ymin = pdf_lognorm(mincon[len(RUL)-1], mu[len(RUL)-1],sigma[len(RUL)-1])
#ymax = pdf_lognorm(maxcon[len(RUL)-1], mu[len(RUL)-1],sigma[len(RUL)-1])
#yfill = [ymin, ymax]
pdf_95 = pdf_lognorm(xfill,mu[len(RUL)-1],sigma[len(RUL)-1])
#plt.fill(xfill, pdf_95, color='b', alpha=.2,label='95% prediction interval')
plt.fill_between(xfill, np.zeros(len(pdf_95)), pdf_95, color='b', alpha=.2,label='95% prediction interval')

#plt.vlines(xfill, ymin=[0,0], ymax=[ymin,ymax], colors='black', lw=2, ls='--', label='95% prediction interval')
plt.xlabel('RUL cycles', fontsize=16)
plt.ylabel('RUL pdf', fontsize=16)
plt.legend(loc='upper right', prop={'size': 12})
#plt.savefig('Images_paper/LSTM_lognorm/Prediction_test_unit_35pdf_2.pdf')

plt.show()

In [ ]:
print(mincon[len(RUL)-1],maxcon[len(RUL)-1])

In [ ]:

y_69 = model.predict(X)
n, p = tf.unstack(pred, num=2, axis=-1)

 # Add one dimension to make the right shape
mu = tf.expand_dims(n, -1)
sigma = tf.expand_dims(p, -1)
y_pred = median_lognorm(mu, sigma).flatten()

l = test_set[:,:,0].flatten()
        # To validate we remove the right padding
y_true = np.ma.compressed(np.ma.masked_where(l==0, l))
y_pred = np.ma.compressed(np.ma.masked_where(l==0, y_pred))

kk = np.diff(test_df0['RUL'].values)
ii = np.where(kk > 0)[0]
index = np.append(ii, 13095)
    
ms = np.exp(mu).flatten()
sigs = np.exp(sigma).flatten()
ms = np.ma.compressed(np.ma.masked_where(l==0, ms))[index]
sigs = np.ma.compressed(np.ma.masked_where(l==0, sigs))[index]
    
mincon = ms/(sigs**2)
maxcon = ms*(sigs**2)
interval =maxcon - mincon
    
ytrue = y_true[index]

be = np.zeros(len(mincon))
be_130 = np.zeros(len(mincon))
for ih in np.arange(len(mincon)):
    if (truth[ih] <= maxcon[ih]) & (truth[ih] >= mincon[ih]):
        be[ih] = 1
    if (ytrue[ih] <= maxcon[ih]) & (truth[ih] >= mincon[ih]):
        be_130[ih] = 1
interval = maxcon-mincon
NMPIW = sum(interval)/(len(mincon)*(max(truth)-min(truth)))
NMPIW_130 = sum(interval)/(len(mincon)*(max(ytrue)-min(ytrue)))
print(be)
print(be_130)
print(sum(be))
print(sum(be_130))
print(NMPIW)   
print(NMPIW_130)

plt.figure(figsize=(16, 7))

# only one line may be specified; full height

# multiple lines with varying ymin and ymax
rangx = np.arange(100)+1
plt.vlines(x=rangx , ymin=mincon, ymax=maxcon, colors='blue', ls='-', lw=1, label='95% prediction interval')
plt.scatter(rangx, truth, s = 50, c = 'red', marker = 'x', label='Ground truth RUL')
plt.scatter(rangx, y_pred[index], s = 30, c = 'black', marker = 'o', label='Mode predicted RUL')
# place legend outside
#plt.legend(bbox_to_anchor=(1.0, 1), loc='upper left')
plt.xlabel('Test unit number', fontsize=16)
plt.ylabel('RUL (cycle)', fontsize=16)
plt.legend(loc='lower left', prop={'size': 12})

plt.savefig('Images_paper/Prediction_results_test.pdf')

plt.show()

In [ ]:
#plt.close()
print("Mean")
ve_tinh(model,
                  [("Train", train_X, train_Y), 
                   ("Validation", val_X, val_Y), 
                   ("Test", test_X, test_Y)],
                 function = mean_lognorm)


In [ ]:
#plt.close()
print("Mode")
ve_tinh(model,
                  [("Train", train_X, train_Y), 
                   ("Validation", val_X, val_Y), 
                   ("Test", test_X, test_Y)],
                 function = mode_lognorm)
#plt.close()

In [14]:
import lime